# 🎯 Bronze to Silver - Loyalty Segments Transformation

## 📊 Objetivo
Transformar dados de segmentos de fidelidade da camada **Bronze** para **Silver**, aplicando limpeza, padronização e validações.

## 🗂️ Tabelas
- **Origem**: `retail_dev.bronze.bronze_loyalty_segments`
- **Destino**: `retail_dev.silver.loyalty_segments`

## 🔄 Transformações Aplicadas
1. **Limpeza**: TRIM e UPPER em descrições
2. **Validação**: Threshold > 0, valid_from/valid_to consistentes
3. **SCD Type 2 Light**: Adicionar flag `is_current` baseado em valid_from
4. **Auditoria**: data_quality_score, processed_at, source_table, pipeline_run_id
5. **Padronização**: Todos os nomes de colunas em UPPER

## 📝 Características
- Tabela de dimensão/lookup (pequena)
- SCD Type 2 simplificado (valid_from/valid_to já existem)
- Delta Lake com Change Data Feed habilitado

In [0]:
from pyspark.sql import functions as F, Window
import uuid
from datetime import datetime

# Configurações
CATALOG = "retail_dev"
BRONZE_SCHEMA = "bronze"
SILVER_SCHEMA = "silver"
BRONZE_TABLE = "bronze_loyalty_segments"
SILVER_TABLE = "loyalty_segments"

# Gerar ID único para este pipeline run
pipeline_run_id = str(uuid.uuid4())
processing_timestamp = datetime.now()

print(f"🔧 Configuração carregada:")
print(f"   📦 Bronze: {CATALOG}.{BRONZE_SCHEMA}.{BRONZE_TABLE}")
print(f"   ✨ Silver: {CATALOG}.{SILVER_SCHEMA}.{SILVER_TABLE}")
print(f"   🆔 Pipeline Run ID: {pipeline_run_id}")
print(f"   ⏰ Timestamp: {processing_timestamp}")

## 📥 Leitura da Camada Bronze

Carregando dados brutos da tabela `bronze_loyalty_segments`.

In [0]:
bronze_table_name = f"{CATALOG}.{BRONZE_SCHEMA}.{BRONZE_TABLE}"

print(f"📖 Lendo tabela: {bronze_table_name}")
bronze_df = spark.table(bronze_table_name)

print(f"\n📊 Total de registros: {bronze_df.count():,}")
print(f"\n📋 Schema da tabela Bronze:")
bronze_df.printSchema()

print(f"\n🔍 Amostra dos dados (5 primeiras linhas):")
display(bronze_df.limit(5))

## 🔍 Auditoria Pré-Transformação

Analisando a qualidade dos dados brutos antes das transformações:
- Verificação de duplicados
- Identificação de valores nulos
- Análise de distribuição de thresholds
- Problemas em valid_to

In [0]:
print("🔍 AUDITORIA PRÉ-TRANSFORMAÇÃO")
print("=" * 80)

# 1. Duplicados por loyalty_segment_id + valid_from
print("\n📊 1. Verificando duplicados (loyalty_segment_id + valid_from):")
duplicates_check = bronze_df.groupBy("loyalty_segment_id", "valid_from").count().filter(F.col("count") > 1)
duplicates_count = duplicates_check.count()
print(f"   ⚠️  Total de duplicados: {duplicates_count:,}")
if duplicates_count > 0:
    print("   📋 Exemplos de duplicados:")
    display(duplicates_check.limit(10))

# 2. Valores nulos
print("\n📊 2. Valores nulos por coluna:")
for col in bronze_df.columns:
    null_count = bronze_df.filter(F.col(col).isNull()).count()
    null_pct = (null_count / bronze_df.count()) * 100 if bronze_df.count() > 0 else 0
    if null_count > 0:
        print(f"   ⚠️  {col}: {null_count:,} ({null_pct:.2f}%)")
    else:
        print(f"   ✅ {col}: 0 (0.00%)")

# 3. Distribuição de unit_threshold
print("\n📊 3. Distribuição de unit_threshold:")
threshold_dist = bronze_df.groupBy("unit_threshold").count().orderBy("unit_threshold")
display(threshold_dist)

# 4. Problemas em valid_to
print("\n📊 4. Análise de valid_to:")
valid_to_nulls = bronze_df.filter(F.col("valid_to").isNull()).count()
valid_to_empty = bronze_df.filter(F.col("valid_to") == "").count()
print(f"   📋 Nulls: {valid_to_nulls:,}")
print(f"   📋 Strings vazias: {valid_to_empty:,}")
print(f"\n   🔍 Valores únicos de valid_to:")
display(bronze_df.select("valid_to").distinct().orderBy("valid_to"))

## 🧹 Limpeza e Padronização

Aplicando transformações:
1. **loyalty_segment_description**: TRIM + UPPER
2. **unit_threshold**: Validar > 0
3. **valid_to**: Limpar valores vazios/inválidos

In [0]:
print("🧹 Aplicando transformações...")

transformed_df = bronze_df.select(
    # IDs mantidos como estão
    F.col("loyalty_segment_id"),
    
    # Description: TRIM + UPPER
    F.trim(F.upper(F.col("loyalty_segment_description"))).alias("loyalty_segment_description"),
    
    # unit_threshold: Validar > 0 (substituir por NULL se <= 0)
    F.when(F.col("unit_threshold") > 0, F.col("unit_threshold")).otherwise(None).alias("unit_threshold"),
    
    # valid_from: manter como DATE
    F.col("valid_from"),
    
    # valid_to: Limpar strings vazias e converter para NULL
    F.when(
        (F.col("valid_to").isNull()) | (F.trim(F.col("valid_to")) == ""),
        None
    ).otherwise(F.col("valid_to")).alias("valid_to")
)

print(f"✅ Transformações aplicadas!")
print(f"\n📊 Total de registros após transformação: {transformed_df.count():,}")
print(f"\n🔍 Amostra (5 primeiras linhas):")
display(transformed_df.limit(5))

## 🔄 Remoção de Duplicados - SCD Type 2 Light

Implementando **Slowly Changing Dimension Type 2 simplificado**:
- Campos `valid_from` e `valid_to` já existem na tabela Bronze
- Adicionar flag `is_current` para indicar o registro ativo
- Lógica: Para cada `loyalty_segment_id`, o registro com `valid_from` mais recente é o atual

In [0]:
print("🔄 Implementando SCD Type 2 Light...")

# Window para ordenar por valid_from DESC dentro de cada loyalty_segment_id
window_spec = Window.partitionBy("loyalty_segment_id").orderBy(F.col("valid_from").desc())

# Adicionar row_number e flag is_current
deduped_df = transformed_df.withColumn(
    "row_num",
    F.row_number().over(window_spec)
).withColumn(
    "is_current",
    F.when(F.col("row_num") == 1, True).otherwise(False)
).drop("row_num")

print(f"✅ SCD Type 2 Light implementado!")
print(f"\n📊 Total de registros: {deduped_df.count():,}")
print(f"\n📊 Distribuição de is_current:")
display(deduped_df.groupBy("is_current").count())

print(f"\n🔍 Amostra (registros correntes):")
display(deduped_df.filter(F.col("is_current") == True).limit(10))

print(f"\n🔍 Exemplo de histórico (um loyalty_segment_id com múltiplas versões):")
example_id = deduped_df.groupBy("loyalty_segment_id").count().filter(F.col("count") > 1).select("loyalty_segment_id").first()
if example_id:
    display(
        deduped_df.filter(F.col("loyalty_segment_id") == example_id[0])
        .orderBy(F.col("valid_from").desc())
    )

## ✅ Validações Finais

Verificando:
1. **Unicidade**: loyalty_segment_id + valid_from deve ser único
2. **Threshold positivo**: unit_threshold > 0
3. **Flag is_current**: Cada loyalty_segment_id tem exatamente 1 registro com is_current = True

In [0]:
print("✅ VALIDAÇÕES FINAIS")
print("=" * 80)

# 1. Unicidade de loyalty_segment_id + valid_from
print("\n📊 1. Verificando unicidade (loyalty_segment_id + valid_from):")
dup_check = deduped_df.groupBy("loyalty_segment_id", "valid_from").count().filter(F.col("count") > 1)
dup_count = dup_check.count()
if dup_count == 0:
    print("   ✅ OK - Todos os registros são únicos")
else:
    print(f"   ❌ ERRO - {dup_count:,} duplicados encontrados")
    display(dup_check.limit(10))

# 2. Threshold positivo
print("\n📊 2. Verificando unit_threshold > 0:")
invalid_threshold = deduped_df.filter(
    (F.col("unit_threshold").isNull()) | (F.col("unit_threshold") <= 0)
).count()
if invalid_threshold == 0:
    print("   ✅ OK - Todos os thresholds são positivos")
else:
    print(f"   ⚠️  {invalid_threshold:,} registros com threshold inválido (NULL ou <= 0)")

# 3. is_current flag
print("\n📊 3. Verificando flag is_current:")
current_per_id = deduped_df.filter(F.col("is_current") == True).groupBy("loyalty_segment_id").count()
invalid_current = current_per_id.filter(F.col("count") != 1).count()

if invalid_current == 0:
    print("   ✅ OK - Cada loyalty_segment_id tem exatamente 1 registro corrente")
else:
    print(f"   ❌ ERRO - {invalid_current:,} loyalty_segment_ids com múltiplos registros correntes")
    display(current_per_id.filter(F.col("count") != 1))

print("\n✅ Validações concluídas!")

## 📊 Adicionar Colunas de Auditoria

Adicionando metadados para rastreabilidade:
- **data_quality_score**: Percentual de campos não-nulos
- **processed_at**: Timestamp do processamento
- **source_table**: Tabela de origem
- **pipeline_run_id**: ID único desta execução

In [0]:
print("📊 Adicionando colunas de auditoria...")

# Campos para cálculo de data_quality_score (5 campos principais)
fields_to_check = [
    "loyalty_segment_id",
    "loyalty_segment_description",
    "unit_threshold",
    "valid_from",
    "is_current"
]

# Calcular data_quality_score
quality_expr = sum([
    F.when(F.col(field).isNotNull(), 1).otherwise(0) for field in fields_to_check
])

final_df = deduped_df.withColumn(
    "data_quality_score",
    (quality_expr / len(fields_to_check) * 100).cast("int")
).withColumn(
    "processed_at",
    F.lit(processing_timestamp)
).withColumn(
    "source_table",
    F.lit(f"{CATALOG}.{BRONZE_SCHEMA}.{BRONZE_TABLE}")
).withColumn(
    "pipeline_run_id",
    F.lit(pipeline_run_id)
)

print(f"✅ Colunas de auditoria adicionadas!")
print(f"\n📊 Schema final (antes da padronização):")
final_df.printSchema()

print(f"\n📊 Distribuição de data_quality_score:")
display(
    final_df.groupBy("data_quality_score")
    .count()
    .orderBy(F.col("data_quality_score").desc())
)

print(f"\n🔍 Amostra (3 primeiras linhas):")
display(final_df.limit(3))

In [0]:
# 🔤 PADRONIZAÇÃO: Converter TODOS os nomes de colunas para UPPER
print("🔤 Padronizando nomes de colunas para UPPER...")
final_df = final_df.select([F.col(c).alias(c.upper()) for c in final_df.columns])
print("✅ Todas as colunas convertidas para UPPER!")

print(f"\n📊 Colunas finais:")
print(final_df.columns)

print(f"\n📋 Schema final:")
final_df.printSchema()

print(f"\n🔍 Amostra (3 primeiras linhas):")
display(final_df.limit(3))

## 💾 Gravar na Camada Silver

Escrevendo dados transformados na tabela Silver:
- **Formato**: Delta Lake
- **Modo**: Overwrite com schema evolution
- **Change Data Feed**: Habilitado
- **Otimização**: OPTIMIZE + ANALYZE TABLE

In [0]:
silver_table_name = f"{CATALOG}.{SILVER_SCHEMA}.{SILVER_TABLE}"

print(f"💾 Gravando na tabela Silver: {silver_table_name}")

# Escrever no formato Delta
final_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .option("delta.enableChangeDataFeed", "true") \
    .saveAsTable(silver_table_name)

print(f"✅ Dados gravados com sucesso!")

# Otimizar tabela
print(f"\n🔧 Otimizando tabela...")
spark.sql(f"OPTIMIZE {silver_table_name}")
print(f"✅ OPTIMIZE concluído!")

# Atualizar estatísticas
print(f"\n📊 Atualizando estatísticas...")
spark.sql(f"ANALYZE TABLE {silver_table_name} COMPUTE STATISTICS")
print(f"✅ ANALYZE TABLE concluído!")

# Verificar tabela criada
print(f"\n📋 Detalhes da tabela Silver:")
silver_df = spark.table(silver_table_name)
print(f"   📊 Total de registros: {silver_df.count():,}")
print(f"   📋 Colunas: {len(silver_df.columns)}")
print(f"\n🔍 Amostra (5 primeiras linhas):")
display(silver_df.limit(5))

## 📈 Relatório Final de Transformação

Sumário completo do pipeline de transformação Bronze → Silver.

In [0]:
print("📈 RELATÓRIO FINAL - LOYALTY SEGMENTS TRANSFORMATION")
print("=" * 80)

# Estatísticas gerais
silver_df = spark.table(silver_table_name)
total_records = silver_df.count()
current_records = silver_df.filter(F.col("IS_CURRENT") == True).count()
historical_records = total_records - current_records

# Recuperar processed_at do DataFrame
processing_timestamp_final = silver_df.select("PROCESSED_AT").first()[0]

# Qualidade dos dados
avg_quality = silver_df.agg(F.avg("DATA_QUALITY_SCORE")).first()[0]
quality_dist = silver_df.groupBy(
    (F.floor(F.col("DATA_QUALITY_SCORE") / 10) * 10).alias("quality_bucket")
).count().orderBy("quality_bucket")

# Distribuição de segmentos
segment_dist = silver_df.filter(F.col("IS_CURRENT") == True) \
    .groupBy("LOYALTY_SEGMENT_DESCRIPTION") \
    .agg(
        F.count("*").alias("count"),
        F.avg("UNIT_THRESHOLD").alias("avg_threshold")
    ).orderBy(F.col("avg_threshold"))

print(f"\n📊 ESTATÍSTICAS GERAIS")
print(f"   Total de registros: {total_records:,}")
print(f"   Registros correntes: {current_records:,}")
print(f"   Registros históricos: {historical_records:,}")
print(f"   Segmentos únicos: {current_records:,}")

print(f"\n📊 QUALIDADE DOS DADOS")
print(f"   Score médio: {avg_quality:.2f}%")
print(f"\n   Distribuição por faixa de qualidade:")
display(quality_dist)

print(f"\n📊 DISTRIBUIÇÃO DE SEGMENTOS (CORRENTES)")
display(segment_dist)

print(f"\n📊 TIMELINE DO SCD TYPE 2")
timeline = silver_df.groupBy("LOYALTY_SEGMENT_ID").agg(
    F.count("*").alias("total_versions"),
    F.min("VALID_FROM").alias("first_valid_from"),
    F.max("VALID_FROM").alias("last_valid_from")
).orderBy(F.col("total_versions").desc())
print("   Segmentos com histórico de mudanças:")
display(timeline.filter(F.col("total_versions") > 1))

# Criar sumário para dashboard
summary_data = spark.createDataFrame([
    ("Pipeline Run ID", str(pipeline_run_id)),
    ("Processed At", str(processing_timestamp_final)),
    ("Source Table", f"{CATALOG}.{BRONZE_SCHEMA}.{BRONZE_TABLE}"),
    ("Destination Table", silver_table_name),
    ("Total Records", str(total_records)),
    ("Current Records", str(current_records)),
    ("Historical Records", str(historical_records)),
    ("Unique Segments", str(current_records)),
    ("Average Quality Score", f"{avg_quality:.2f}%")
], ["Metric", "Value"])

print(f"\n📊 SUMÁRIO EXECUTIVO")
display(summary_data)

print(f"\n✅ Pipeline concluído com sucesso! 🎉")